# Soft tissue mechanics in FEniCSx

**Joakim Sundnes**

Date: **June 20, 2025**


* Solve a simple soft tissue mechanics problem:
  * Elastic unit cube
  * Uniaxial stretch
  * St Venant-Kirchhoff material


## Recap of key quantities 
* $u(t) = x(t) - X$: Displacement, usually the primary unknown of our mechanics problems. 
* ${F} = {I} + \nabla{u}$: Deformation gradient. Used in strain definitions and to map between deformed and undeformed geometries.
* ${C} = {F}^T{F}$: Right Cauchy-Green tensor. Measure of pure deformation, free of rigid body translation and rotation, often used in constitutive laws. Not technically strain since ${C} = {I}$ for zero displacement.
* ${E} = 1/2({F}^T{F}-{I}) = 1/2({C}-{I})$: Green-Lagrange strain. The most commonly used strain tensor for large deformations. Extension of the standard, linear small deformation strain.
* ${\sigma}$: Cauchy stress tensor, also called the true stress. Standard stress tensor used for small deformations. Relevant also for large deformations, but inconvenient to use in computations. 
* ${P}, {S}$: First and second Piola-Kirchhoff stress tensors. Stress tensors suitable for large deformations, since they are defined relative to the undeformed geometry. 
* $W$: Strain energy function. Defines the elastic energy stored during a deformation. Used to define material laws, since stress is recovered as ${S} = \partial W/\partial{E}$, etc.



## Example 1: An elastic cube
We start with a simple example, where we consider loading of a unit cube. This simple case will illustrate how to define a non-linear elasticity problem in FEniCSx, and how to introduce boundary conditions and different material properties. The model will be a simple unit cube, fixed at one end ($x=0$) and loaded with a pressure load (stretch) at the other end ($x=1.0$). The first version of the model will use a simple isotropic neo-Hookean material model. Later we will introduce a more realistic soft tissue model, and also add active contraction. 


### Weak form of the static hyper-elasticity problem
We want to solve a static solid mechanics problem with a combination of Dirichlet and Neumann boundary conditions:
$$
 \begin{alignat*}{2}
    - \nabla\cdot P &= 0 && \quad \text{ in } \Omega \\
    u &= 0 && \quad \text{ on } \Gamma_{\mathrm{D}} \\
    P \cdot n &= T && \quad \text{ on } \Gamma_{\mathrm{N}} \\
    P \cdot n &= 0 && \quad \text{ on } \Gamma_{\mathrm{0}}
  \end{alignat*}
$$
Here, $P$ is the first Piola-Kirchhoff stress tensor, $u$ is the displacement, $T$ is a load (vector) applied to parts of the boundary, and $\Omega, \Gamma_{\mathrm{D}}, \Gamma_{\mathrm{N}},\Gamma_{\mathrm{0}}$ are the domain and the boundaries for Dirichlet- and Neumann boundary conditions, respectively. 

To apply the finite element method, we need to derive the weak form of the problem.  Multiply by a test function $v \in \hat{V}$ and integrate by parts:
$$
  \begin{equation*}
    - \int_{\Omega} \nabla\cdot P \cdot v dx
    = \int_{\Omega} P : \nabla v dx - \int_{\partial\Omega} (P \cdot n) \cdot v ds = 0
  \end{equation*}
$$

We now apply the boundary conditions (and note that $v = 0$ on $\Gamma_{\mathrm{D}}$), to get the final weak form:

Find $u \in V$ such that
$$
\begin{equation*}
    \int_{\Omega} P : \nabla v dx
    = \int_{\Gamma_{\mathrm{N}}} T \cdot v ds
  \end{equation*}
$$
for all $v \in \hat{V}$.

In our case, the boundary load $T$ is a simple normal pressure. If we were working in the deformed configuration, we would simply have $T=-p n$, where $p$ is the applied pressure and $n$ is the unit surface normal. However, we are using a Lagrangian formulation and everything is defined relative to the reference state. The applied pressure, which actually acts on the deformed surface, therefore needs to be mapped to the reference state. We get
$$
\begin{equation*}
T = -p J F^{-T}\cdot N,
\end{equation*}
$$
where $N$ is the unit normal of the undeformed (reference) geometry. 

### The St Venant-Kirchhoff material model
The simplest hyper-elastic material model is the St Venant-Kirchhoff model, which is simply an extension of the linear Hooke's law to the large-deformation regime. The strain energy function is given by:
$$
\begin{equation*}
    W(E) = \frac{\lambda}{2} (\operatorname{tr}(E))^2 + \mu \operatorname{tr}(E^2),
\end{equation*}
$$
where $E = \frac{1}{2}(F^T F - I)$ is the Green-Lagrange strain, and $F = I + \nabla u$ is the deformation gradient. From the strain energy function we can obtain the first and second Piola-Kirchhoff (PK) stress ($P$ and $S$) as
$$
\begin{align*}
P_{ij} &= \frac{\partial W}{\partial F_{ij}}, \\
S_{ij} &= \frac{\partial W}{\partial E_{ij}},
\end{align*}
$$
and we also have the relation $P = FS$. Both the first and second PK stresses are commonly used in finite element solvers. The formulation based on $S$ is popular in text books, as it allows a few more analytical steps in the derivation of the nonlinear solution method. However, in a computational setting, and in particular when using an automated system like FEniCSx, the formulation based on $P$ is just as simple.

### The FEniCSx solver
We are now ready to specify the problem in FEniCSx. First, the usual imports and defining the solution domain, the function space, and the test- and trial functions:

In [ ]:
from dolfinx import fem, geometry, mesh, io, plot, default_scalar_type
import dolfinx.fem.petsc
from ufl import (
    TestFunction,
    Measure,
    FacetNormal,
    variable,
    Identity,
    grad,
    diff,
    dot,
    inner,
    tr,
    det,
    inv,
    dx,
)
from mpi4py import MPI
from matplotlib import pyplot as plt
import numpy as np
from plotting import setup_gif_visualizer, update_gif_frame

# Create the mesh and the function space for the solutions
domain = mesh.create_unit_cube(MPI.COMM_WORLD, 4, 4, 4)
V = fem.functionspace(domain, ("CG", 2, (domain.geometry.dim,)))

# Define functions
v = TestFunction(V)  # Test function
u = fem.Function(V, name="u")  # Displacement

In [ ]:
# Plot the mesh with Pyvista
import pyvista

pyvista.set_jupyter_backend("static")

pv_grid = pyvista.UnstructuredGrid(*plot.vtk_mesh(domain))
plotter = pyvista.Plotter(window_size=[300, 300])
plotter.add_mesh(pv_grid, show_edges=True)
plotter.show()

Now we need to define the boundary conditions. We want homogenous Dirichlet conditions on the left boundary ($x=0.0$), a non-homogenous Neumann condition on the right boundary ($x=1.0$), and homogenous Neumann conditions everywhere else. The following code first defines the subdomains and marks the respective boundaries, then redefines the boundary measure (`ds`) to allow surface integrals over parts of the boundary, and finally defines the Dirichlet conditions:

In [ ]:
# Mark boundary subdomains
fdim = domain.topology.dim - 1  # facet dimension


# Define marker functions for the left and right boundaries
def left(x):
    return np.isclose(x[0], 0)


def right(x):
    return np.isclose(x[0], 1.0)


# Locate the degrees of freedom and facets on the left and right boundaries
dofs_l = fem.locate_dofs_geometrical(V, left)
facets_l = mesh.locate_entities(domain, fdim, left)

dofs_r = fem.locate_dofs_geometrical(V, right)
facets_r = mesh.locate_entities(domain, fdim, right)

# Generate the meshtags, tagging the left and right boundaries with different markers
marker_l = 1
marker_r = 2

entities = np.hstack([facets_l, facets_r])
values = np.hstack([np.full_like(facets_l, marker_l), np.full_like(facets_r, marker_r)])
boundary_markers = mesh.meshtags(
    domain,
    fdim,
    entities,
    values,
)

# Redefine boundary measure
ds = Measure("ds", domain, subdomain_data=boundary_markers)

# Define Dirichlet boundary condition on left boundary
bc = fem.dirichletbc(np.zeros(3, dtype=default_scalar_type), dofs_l, V)
bcs = [bc]

We also define a point $(1.0, 0.5, 0.5)$ where we will track the $x$ component of the displacement $u$. 

In [ ]:
track_point = [1.0, 0.5, 0.5]

def evaluate_at_point(mesh, u, point):
    pt_array = np.array([point], dtype=mesh.geometry.x.dtype)
    ownership = geometry.determine_point_ownership(mesh, pt_array, padding=1e-6)
    cells = ownership.dest_cells
    cell_idx = cells[0]
    value = u.eval(pt_array, np.array([cell_idx], dtype=np.int32))
    return value

Next, we turn to defining the mechanics problem. We start with the kinematics and the strain energy function defining the St Venant-Kirchhoff material, obtain the Piola-Kirchhoff stresses by differentiating the strain energy function, and finally define the weak form of the problem.

### Exercise 

Continue the code below to define all the relevant kinematic quantities, define the weak form, and then solve the problem.  

In [ ]:
# Kinematics
d = len(u)
I = Identity(d)  # Identity tensor
F = I + grad(u)  # Deformation gradient

# TODO: Define the remaining kinetic quantities and strain energy function

p_right = fem.Constant(domain, 0.0)  # the pressure load (zero for now)

# TODO: Define residual
# R = ...

# Set up nonlinear problem
petsc_options = {
    "ksp_type": "preonly",
    "pc_type": "lu",
    "pc_factor_mat_solver_type": "mumps",
    "snes_monitor": None,
}
problem = fem.petsc.NonlinearProblem(
    R,
    u,
    bcs=bcs,
    petsc_options=petsc_options,
    petsc_options_prefix="nonlinear_basic",
)

# Finally, we solve the problem for different loads, and plot the load vs displacement.

# Step-wise loading (for plotting and convergence)
load_steps = 5
target_load = 10.0
loads = np.linspace(0, target_load, load_steps)
disps = np.zeros(load_steps)

for step in range(load_steps):
    print(f"Solving for load={loads[step]}")

    # Update traction value (stretch is a negative pressure)
    p_right.value = -loads[step]

    # Solve the nonlinear problem
    problem.solve()

    # Evaluate displacement at point defined above
    disps[step] = evaluate_at_point(domain, u, track_point)[0] # extract x comp.


# TODO: Plot loads vs displacements

## Solution
Click below to expand the cell and see a suggested solution. This code also writes the displacement to file (`output/passive_cube_u.bp`) and generates a GIF showing the cube deformation (`output/passive_cube.gif`).

In [ ]:
# Kinematics
d = len(u)
I = variable(Identity(d))  # Identity tensor
F = variable(I + grad(u))  # Deformation gradient
C = variable(F.T * F)  # Right Cauchy-Green tensor
E = variable(0.5 * (C - I))  # Green-Lagrange strain tensor

# Material parameters (Lamé parameters)
mu = 4.0
lmbda = 20.0

# The strain energy for the St-Venant Kirchhoff model:
psi = lmbda / 2 * (tr(E) ** 2) + mu * tr(E * E)

S = diff(psi, E)  # Second Piola-Kirchhoff stress
P = F * S  # First Piola-Kirchhoff stress (alt. P = diff(psi, F))

p_right = fem.Constant(domain, 0.0)  # the pressure load (zero for now)

# Definition of the weak form:
N = FacetNormal(domain)
traction = -p_right * det(F) * dot(inv(F).T, N)
# Residual: Internal forces - External forces
R = inner(grad(v), P) * dx - inner(v, traction) * ds(2)

# Set up nonlinear problem
petsc_options = {
    "ksp_type": "preonly",
    "pc_type": "lu",
    "pc_factor_mat_solver_type": "mumps",
    "snes_monitor": None,
}
problem = fem.petsc.NonlinearProblem(
    R,
    u,
    bcs=bcs,
    petsc_options=petsc_options,
    petsc_options_prefix="nonlinear_basic",
)

# Prepare output file
outfile = io.VTXWriter(domain.comm, "output/passive_cube_u.bp", [u])
outfile.write(0.0)
# Prepare GIF
plotter, grid, magnitude, us_expr, actor = setup_gif_visualizer(
    domain, u, filename="output/passive_cube.gif"
)

# Finally, we solve the problem for different loads, and plot the load vs displacement.

# Step-wise loading (for plotting and convergence)
load_steps = 5
target_load = 10.0
loads = np.linspace(0, target_load, load_steps)
disps = np.zeros(load_steps)

for step in range(load_steps):
    print(f"Solving for load={loads[step]}")

    # Update traction value
    p_right.value = -loads[step]

    # Solve the nonlinear problem
    problem.solve()

    # Evaluate displacement at point defined above
    disps[step] = evaluate_at_point(domain, u, track_point)[0] # extract x comp.

    # Write GIF frame
    update_gif_frame(plotter, grid, u, magnitude, us_expr, actor)

    # Write displacement to file
    outfile.write(loads[step])

outfile.close()
plotter.close()

plt.figure()
plt.plot(loads, disps, ".-")
plt.xlabel("Applied pressure load")
plt.ylabel(r"Displacement $u_x$ of point (1.0, 0.5, 0.5)")
plt.show()

In [ ]:
# Display the generated GIF
from IPython.display import Image

Image(filename="output/passive_cube.gif", width=500)

### Questions and comments:
* The St Venant-Kirchhoff model is a linear stress-strain relation, but the curve above is non-linear. Why?
* An open source cardiac mechanics solver, based on the approach outlined above, can be found here: [https://github.com/finsberg/pulse](https://github.com/finsberg/pulse) (Legacy FEniCS) / [https://github.com/finsberg/fenicsx-pulse](https://github.com/finsberg/fenicsx-pulse) (FEniCSx)